## List of datasets to create

All datasets to have 0.93 visibility, 0.08 degree misalignment, 10000 states
For each folder of different measurements, have data for 10,20,40,80,160,320,640,1280 shots, and for each of these have mixed and pure states (so 2*8 = 16 total for each overall setting)
Do this for 2 and 3 qubits
Create the test train split dictionaries and store those instead of the large one

In [13]:
import numpy as np
import QST_core_processes as qst

In [14]:
n_qubits = [2,3]
p_pure = [0,1]
n_shots = [10, 20, 40, 80 ,160, 320, 640, 1280]

In [15]:
def generate_and_store(n_qubits, p_pure, n_shots):
    
    N = 10000
    SEED = 53987

    rhos, taus = qst.generate_dataset_of_states_and_probabilities(N, n_qubits, SEED, p_pure, eps_pure=1e-5)
    data = {'rhos': rhos, 'taus': taus}

    P = qst.build_projector_matrix(n_qubits)
    P_noisy = qst.simulate_waveplate_misalignment(P, n_qubits)

    counts = []
    for rho in data['rhos']:
        p = qst.get_measurement_probs_from_P_and_rho(rho, P_noisy, n_qubits)
        p_noisy = qst.simulate_interference_visibility(p)
        counts.append(qst.retrieve_counts_from_n_shots_per_state(p_noisy, n_shots))

    data['counts'] = np.stack(counts, axis=0) # easier to work with as a single array
    data['P'] = P
    data['P_noisy'] = P_noisy
    data['shots'] = n_shots

    qst.add_train_test_split_to_data(data, train_ratio=0.9, seed=SEED)
    data_train = qst.get_split(data, "train")
    data_test  = qst.get_split(data, "test")

    if p_pure == 0:
        p_pure_str = "mixed"
    else:
        p_pure_str = "pure"

    string = f"{n_qubits}q_{p_pure_str}_{n_shots}shots"
    if p_pure == 0.5:
        string = "hyperparam_tuning_set"

    np.savez_compressed(f"data/{string}_train.npz", **data_train)
    np.savez_compressed(f"data/{string}_test.npz", **data_test)

In [16]:
for n in n_qubits:
    for p in p_pure:
        for s in n_shots:
            generate_and_store(n, p, s)
            print(f"Generated and stored dataset for {n} qubits, p_pure={p}, n_shots={s}")

Generated and stored dataset for 2 qubits, p_pure=0, n_shots=10
Generated and stored dataset for 2 qubits, p_pure=0, n_shots=20
Generated and stored dataset for 2 qubits, p_pure=0, n_shots=40
Generated and stored dataset for 2 qubits, p_pure=0, n_shots=80
Generated and stored dataset for 2 qubits, p_pure=0, n_shots=160
Generated and stored dataset for 2 qubits, p_pure=0, n_shots=320
Generated and stored dataset for 2 qubits, p_pure=0, n_shots=640
Generated and stored dataset for 2 qubits, p_pure=0, n_shots=1280
Generated and stored dataset for 2 qubits, p_pure=1, n_shots=10
Generated and stored dataset for 2 qubits, p_pure=1, n_shots=20
Generated and stored dataset for 2 qubits, p_pure=1, n_shots=40
Generated and stored dataset for 2 qubits, p_pure=1, n_shots=80
Generated and stored dataset for 2 qubits, p_pure=1, n_shots=160
Generated and stored dataset for 2 qubits, p_pure=1, n_shots=320
Generated and stored dataset for 2 qubits, p_pure=1, n_shots=640
Generated and stored dataset for

In [11]:
generate_and_store(2, 1, 320)